# 🫧 K-Means Clustering
**ISLP Chapter 12 · Data Pattern Recognition for the Rest of Us**

> K-means groups customers into segments by finding cluster centers that minimize within-cluster distance. The result: actionable customer segments you can market to differently.

### Dataset
**E-commerce Customer Segmentation via RFM Analysis**
RFM (Recency, Frequency, Monetary) is the gold standard framework for customer analytics. We build it from a UK online retail transaction dataset and cluster customers into distinct behavioral segments.

### Time: ~55 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
print("\u2713 Setup complete")

In [ ]:
# UK Online Retail dataset — build RFM features
# Source: UCI ML Repository — real e-commerce transactions 2010-2011
import numpy as np
import pandas as pd

try:
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
    retail = pd.read_excel(url, engine='openpyxl')
    retail = retail.dropna(subset=['CustomerID'])
    retail = retail[retail['Quantity'] > 0]
    retail['TotalPrice'] = retail['Quantity'] * retail['UnitPrice']
    snapshot_date = retail['InvoiceDate'].max()
    rfm = retail.groupby('CustomerID').agg(
        Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
        Frequency=('InvoiceNo', 'nunique'),
        Monetary=('TotalPrice', 'sum')
    ).reset_index()
    print("Loaded UK Online Retail dataset from UCI")
except:
    # Synthetic RFM — realistic e-commerce customer distribution
    np.random.seed(42)
    n = 4372  # matches real dataset customer count
    # Champions: low recency, high freq, high spend
    n_champ = 500
    # At-Risk: high recency, was good
    n_risk = 800
    # Hibernating: very high recency, low freq
    n_hiber = 1200
    # New: low recency, low freq
    n_new = 600
    # Loyal: medium recency, high freq
    n_loyal = 700
    # Others
    n_other = n - n_champ - n_risk - n_hiber - n_new - n_loyal

    rfm_data = []
    for seg, nr, rec_mu, rec_sd, freq_mu, freq_sd, mon_mu, mon_sd in [
        ('Champions',   n_champ, 15,  10, 12,  4,  1500, 600),
        ('At-Risk',     n_risk,  120, 40, 8,   3,  900,  400),
        ('Hibernating', n_hiber, 280, 60, 2,   1,  200,  100),
        ('New',         n_new,   20,  12, 1,   0.5,150,   80),
        ('Loyal',       n_loyal, 45,  20, 10,  3,  1100, 450),
        ('Others',      n_other, 180, 70, 3,   2,  350,  200),
    ]:
        rfm_data.append(pd.DataFrame({
            'CustomerID': range(len(rfm_data)*1000, len(rfm_data)*1000+nr),
            'Recency':    np.abs(np.random.normal(rec_mu, rec_sd, nr)).astype(int).clip(1,365),
            'Frequency':  np.abs(np.random.normal(freq_mu,freq_sd,nr)).astype(int).clip(1,50),
            'Monetary':   np.abs(np.random.normal(mon_mu, mon_sd, nr)).clip(10, 10000).round(2)
        }))
    rfm = pd.concat(rfm_data, ignore_index=True)
    print("Using synthetic RFM dataset (realistic UK retail properties)")

print(f"\nRFM dataset: {rfm.shape[0]:,} customers")
print(f"\nRFM Summary:")
print(rfm[['Recency','Frequency','Monetary']].describe().round(1).to_string())
print("\nInterpretation:")
print("  Recency:   days since last purchase (lower = better)")
print("  Frequency: number of distinct orders (higher = better)")
print("  Monetary:  total spend in GBP (higher = better)")

## 📐 Part 1 — K-Means Algorithm

At each iteration:
1. **Assign** each point to its nearest centroid (minimize Euclidean distance)
2. **Update** each centroid to the mean of its assigned points
3. Repeat until assignments stop changing

**Key property:** K-means minimizes **Within-Cluster Sum of Squares (WCSS)**.
Always standardize first — monetary values in £ would completely dominate recency in days.

In [ ]:
# Standardize RFM (critical — Monetary in £ vs Recency in days otherwise)
X_rfm = rfm[['Recency','Frequency','Monetary']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rfm)

# Note: we invert Recency so higher always means "better" for interpretation
X_scaled[:,0] *= -1  # higher score = more recent

# Elbow method — find optimal K
wcss = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_range), wcss, 'o-', color='#1e5fa8', lw=2, markersize=7)
axes[0].set_xlabel('Number of Segments (K)')
axes[0].set_ylabel('WCSS (Within-Cluster Sum of Squares)')
axes[0].set_title('Elbow Method\n(look for the "elbow" — diminishing returns)')

axes[1].plot(list(K_range), sil_scores, 's-', color='#e85d20', lw=2, markersize=7)
best_k = list(K_range)[sil_scores.index(max(sil_scores))]
axes[1].axvline(best_k, color='#1a7a45', lw=1.5, ls='--',
                label=f'Best K={best_k} (silhouette={max(sil_scores):.3f})')
axes[1].set_xlabel('Number of Segments (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score — higher = more distinct segments')
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"\n\u2714 Silhouette suggests K={best_k} customer segments")

In [ ]:
# Fit final model and name the segments
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=20)
rfm['Segment'] = km_final.fit_predict(X_scaled)

# Profile each segment on original scale
seg_profile = rfm.groupby('Segment').agg(
    N=('CustomerID','count'),
    Avg_Recency=('Recency','mean'),
    Avg_Frequency=('Frequency','mean'),
    Avg_Monetary=('Monetary','mean')
).round(1)
seg_profile['% of customers'] = (seg_profile['N'] / len(rfm) * 100).round(1)

# Name segments based on profile
def name_segment(row):
    if row['Avg_Recency'] < 40 and row['Avg_Frequency'] > 7:
        return 'Champions'
    elif row['Avg_Recency'] < 60 and row['Avg_Frequency'] > 4:
        return 'Loyal Customers'
    elif row['Avg_Recency'] > 150:
        return 'Lost / Hibernating'
    elif row['Avg_Frequency'] <= 2 and row['Avg_Recency'] < 50:
        return 'New Customers'
    else:
        return 'At-Risk'

seg_profile['Segment Name'] = seg_profile.apply(name_segment, axis=1)
print("=== Customer Segment Profiles ===\n")
print(seg_profile[['Segment Name','N','% of customers',
                    'Avg_Recency','Avg_Frequency','Avg_Monetary']].to_string())

In [ ]:
# Visualize segments in 2D (PCA)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

seg_names = rfm['Segment'].map(
    seg_profile['Segment Name'].to_dict())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_seg = ['#1e5fa8','#e85d20','#1a7a45','#6b46c1','#0e7490']

for seg_id in rfm['Segment'].unique():
    mask = rfm['Segment'] == seg_id
    name = seg_profile.loc[seg_id,'Segment Name']
    axes[0].scatter(X_pca[mask,0], X_pca[mask,1], s=12, alpha=0.5,
                   color=colors_seg[seg_id % len(colors_seg)], label=name)

axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% variance)')
axes[0].set_title('Customer Segments in PCA Space')
axes[0].legend(fontsize=9)

# Radar/bar chart of segment profiles
seg_profile_norm = seg_profile[['Avg_Recency','Avg_Frequency','Avg_Monetary']].copy()
for col in seg_profile_norm.columns:
    seg_profile_norm[col] = (seg_profile_norm[col] - seg_profile_norm[col].min()) /                              (seg_profile_norm[col].max() - seg_profile_norm[col].min())
seg_profile_norm.index = seg_profile['Segment Name']

seg_profile_norm.plot(kind='bar', ax=axes[1], color=['#1e5fa8','#e85d20','#1a7a45'],
                      edgecolor='white', width=0.7)
axes[1].set_title('Normalized Segment Profiles\n(higher = better for Frequency/Monetary, lower = better for Recency)')
axes[1].set_xlabel('Segment')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()
print("\n\u2714 Business actions by segment:")
print("  Champions:        Reward loyalty — offer VIP program")
print("  Loyal Customers:  Upsell — they're engaged and spending")
print("  At-Risk:          Re-engagement campaign — discount or win-back email")
print("  New Customers:    Onboarding nurture — encourage 2nd purchase")
print("  Lost/Hibernating: Low-cost reactivation attempt or write off")

In [ ]:
answers = {
    "q1": "",  # What does K-means minimize, and why must features be standardized first?
    "q2": "",  # What does the silhouette score measure? What does a value near 0 mean?
    "q3": "",  # What is the elbow method and why doesn't it always give a clear answer?
    "q4": "",  # Why is K-means sensitive to initialization, and how does n_init help?
    "q5": "",  # For each customer segment identified, describe ONE concrete marketing action.
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("\u274c Still empty: "+str(missing) if missing else "\u2705 Done! File \u2192 Save a copy in GitHub")

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*